# Foodbank Agent — Colab Debug Notebook

Use this notebook to:
- **Test queries** against the real agent and data without touching the Render app
- **Debug failures** — see the LLM-generated code and any errors inline
- **Launch a live Streamlit preview** via a public tunnel

---
## How to use
1. Add your secrets in **Colab → 🔑 Secrets** (left sidebar)
2. Run cells top to bottom
3. Section 2 lets you test queries inline (fastest — no browser needed)
4. Section 5 launches the full Streamlit UI via a public URL

---
## 0 · Setup
Clone the repo and install dependencies. Run once per session.

In [ ]:
import os

REPO_DIR = "/content/Foodbank_agent"

if not os.path.exists(REPO_DIR):
    !git clone -b pandas-code-gen https://github.com/spykeshivam/Foodbank_agent.git
else:
    !cd {REPO_DIR} && git pull origin pandas-code-gen

%cd {REPO_DIR}
print("Repo ready:", os.getcwd())

In [ ]:
!pip install -q openai pandas plotly openpyxl python-dateutil gspread python-dotenv streamlit
print("Dependencies installed")

### Secrets

Add these in **Colab → 🔑 Secrets** (the key icon in the left sidebar):

| Secret name | Where to find it |
|---|---|
| `GROQ_API_KEY` | console.groq.com |
| `CEREBRAS_API_KEY` | cloud.cerebras.ai |
| `GOOGLE_CREDENTIALS_B64` | Render dashboard → env vars |
| `SHEET_ID` | Render dashboard → env vars |
| `LOGIN_SHEET_ID` | Render dashboard → env vars |
| `APP_USERNAME` | Render dashboard → env vars |
| `APP_PASSWORD` | Render dashboard → env vars |

In [ ]:
from google.colab import userdata

REQUIRED = [
    "GROQ_API_KEY",
    "CEREBRAS_API_KEY",
    "GOOGLE_CREDENTIALS_B64",
    "SHEET_ID",
    "LOGIN_SHEET_ID",
    "APP_USERNAME",
    "APP_PASSWORD",
]

missing = []
for key in REQUIRED:
    try:
        os.environ[key] = userdata.get(key)
        print(f"  ✓ {key}")
    except Exception:
        missing.append(key)
        print(f"  ✗ {key}  ← missing")

if missing:
    print("\n⚠️  Add the missing secrets in Colab → 🔑 Secrets, then re-run this cell.")
else:
    print("\nAll secrets loaded.")

---
## 1 · Load Data
Fetch live data from Google Sheets and inspect it.

In [ ]:
import sys

import pandas as pd

sys.path.insert(0, REPO_DIR)

from sheets import fetch_logins, fetch_registrations

print("Fetching data from Google Sheets...")
registrations = pd.DataFrame(fetch_registrations())
logins = pd.DataFrame(fetch_logins())

print(f"  registrations: {registrations.shape[0]:,} rows × {registrations.shape[1]} cols")
print(f"  logins:        {logins.shape[0]:,} rows × {logins.shape[1]} cols")

In [ ]:
# Inspect registrations columns and sample rows
print("Registrations columns:")
for col in registrations.columns:
    print(f"  {col}: {registrations[col].dtype}")
registrations.head(3)

In [ ]:
# Inspect logins columns and sample rows
print("Logins columns:")
for col in logins.columns:
    print(f"  {col}: {logins[col].dtype}")
logins.head(3)

---
## 2 · Test Queries Inline

Calls the agent directly — no Streamlit needed.
You see the result (text, table, chart) and any errors, right in the notebook.

In [ ]:
import plotly.io as pio
from IPython.display import display

from agent import PROVIDERS, run_query


def ask(question: str, provider: str = "Groq"):
    """Run a question through the agent and display results inline."""
    print(f"Question : {question}")
    print(f"Provider : {provider} ({PROVIDERS[provider]['model']})")
    print("-" * 60)

    result = run_query(question, registrations, logins, provider=provider)

    if result.error:
        print(f"ERROR:\n{result.error}")
        return result

    for block in result.display_blocks:
        if block["type"] == "text":
            print(block["text"])
        elif block["type"] == "dataframe":
            display(block["data"])
        elif block["type"] == "chart":
            with open(block["path"]) as f:
                fig = pio.from_json(f.read())
            fig.show()

    return result

In [ ]:
result = ask("How many people have Halal dietary requirements?")

In [ ]:
result = ask("Show me the gender breakdown as a pie chart.")

In [ ]:
result = ask("How many logins were there each month? Show a bar chart.")

In [ ]:
# ── Your own question — edit and run ─────────────────────────────────────────
result = ask("YOUR QUESTION HERE")

---
## 3 · Debug Mode

Use this when a query gives wrong answers or errors.
It shows the exact code the LLM generated, and any tracebacks.

In [ ]:
from agent import _build_client, _build_messages, _call_llm, _execute, _extract_code


def debug_query(question: str, provider: str = "Groq"):
    """Step through the agent pipeline and print every intermediate value."""
    print(f"=== DEBUG: {question!r} ({provider}) ===")

    client, model = _build_client(provider)
    messages = _build_messages(question, registrations, logins, history=[])

    print("\n── System prompt (first 500 chars) ──")
    print(messages[0]["content"][:500], "...")

    print(f"\n── Calling {provider} ({model}) ──")
    raw = _call_llm(client, model, messages)

    print("\n── Raw LLM response ──")
    print(raw)

    code = _extract_code(raw)
    print("\n── Extracted code ──")
    print(code)

    output, error = _execute(code, registrations, logins)

    if error:
        print("\n── Execution error ──")
        print(error)
    else:
        print(f"\n── Output keys: {list(output.keys())} ──")
        if "result" in output:
            display(output["result"])
        if "fig" in output:
            output["fig"].show()

    return raw, code, output, error

In [ ]:
# Run debug on any query that's giving trouble
raw, code, output, error = debug_query("How many logins were there each month?")

---
## 4 · Batch Smoke Test

Run a standard set of queries against one or both providers and report pass / fail.
Run this after any change to `agent.py` before pushing.

In [ ]:
import time

SMOKE_QUERIES = [
    "How many people have Halal dietary requirements?",
    "Show me the gender breakdown as a pie chart.",
    "How many logins were there each month? Show a bar chart.",
    "How many male users visited on a Tuesday?",
    "What are the top 5 spoken languages?",
]


def smoke_test(provider: str):
    print(f"\n{'='*60}")
    print(f"Provider: {provider} — {PROVIDERS[provider]['model']}")
    print("=" * 60)
    results = []
    for i, q in enumerate(SMOKE_QUERIES):
        if i > 0:
            time.sleep(5)  # avoid rate limits
        try:
            r = run_query(q, registrations, logins, provider=provider)
            if r.error:
                status = f"FAIL  — {r.error[:80]}"
            elif not r.display_blocks:
                status = "FAIL  — no output"
            else:
                types = [b["type"] for b in r.display_blocks]
                status = f"PASS  — {types}  |  {r.text[:60]}"
        except Exception as e:
            status = f"EXCEPTION — {e}"
        icon = "✓" if status.startswith("PASS") else "✗"
        print(f"  {icon}  {q[:50]:<50}  {status}")
        results.append(status.startswith("PASS"))
    print(f"\n  {sum(results)}/{len(results)} passed")
    return results


groq_results = smoke_test("Groq")

In [ ]:
# Run for Cerebras (comment out if you don't have the key)
cerebras_results = smoke_test("Cerebras")

---
## 5 · Launch Streamlit App

Starts the full Streamlit UI and gives you a public URL via a Cloudflare tunnel.
No Cloudflare account needed. The URL is temporary — lives as long as this Colab session.

In [ ]:
# Install cloudflared (no account needed)
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o /tmp/cloudflared.deb
!dpkg -i /tmp/cloudflared.deb 2>/dev/null || true
print("cloudflared ready")

In [ ]:
import re
import subprocess
import time

PORT = 8501

# Start Streamlit
streamlit_proc = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        f"--server.port={PORT}",
        "--server.address=0.0.0.0",
        "--server.headless=true",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(4)

# Start cloudflared and capture the public URL
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stderr=subprocess.PIPE,
    stdout=subprocess.DEVNULL,
)

public_url = None
for line in tunnel_proc.stderr:
    decoded = line.decode("utf-8", errors="ignore")
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", decoded)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print(f"🌐 Streamlit app: {public_url}")
    print("   (URL is live until you stop this cell or close the session)")
else:
    print("⚠️  Could not get tunnel URL — check cloudflared is installed")

In [ ]:
# Stop Streamlit + tunnel when done
streamlit_proc.terminate()
tunnel_proc.terminate()
print("Stopped.")